# 01 · CRM Data Cleaning and Preparation

**Project:** CRM analysis of online programming school X (July 2023 – June 2024).
**Goal of this notebook:** turn 4 raw CRM exports into clean, linked tables
that support funnel, marketing-efficiency and unit-economics analysis.

| Table | Contents | Key |
|---|---|---|
| Contacts | contacts (leads) | `Id` |
| Calls | sales managers' calls | `CONTACTID` → Contacts |
| Spend | daily ad spend | `Source`, `Campaign` → Deals |
| Deals | potential deals | `Contact Name` → Contacts |

All cleaning logic lives in the module [`src/cleaning.py`](../src/cleaning.py) so it can be
reused and tested. This notebook checks the raw data, runs the ETL and validates the result.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import cleaning as etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Raw data

We read `.xlsx`, not `.csv`: in the CSV export the Cyrillic text in *Level of Deutsch* turned into `?`,
and 19-digit IDs were rounded as floats (which breaks the joins). All IDs are read as strings.

In [2]:
raw = etl.load_raw()
overview = pd.DataFrame({
    name: {"rows": len(df), "columns": df.shape[1],
           "full duplicates": df.duplicated().sum(),
           "missing cells, %": round(df.isna().mean().mean() * 100, 1)}
    for name, df in raw.items()
}).T
overview

,rows,columns,full duplicates,"missing cells, %"
Contacts,18548.0,4.0,0.0,0.0
Calls,95874.0,11.0,0.0,20.3
Spend,20779.0,8.0,917.0,11.8
Deals,21595.0,23.0,0.0,43.0


In [3]:
def missing_report(df):
    m = df.isna().sum()
    return (pd.DataFrame({"missing": m, "%": (m / len(df) * 100).round(1),
                          "dtype": df.dtypes.astype(str)})
            .query("missing > 0").sort_values("%", ascending=False))

missing_report(raw["Deals"])

,missing,%,dtype
Payment Type,21099,97.7,str
Months of study,20755,96.1,float64
Level of Deutsch,20344,94.2,object
City,19084,88.4,str
Education Type,18295,84.7,str
Course duration,18008,83.4,float64
Product,18003,83.4,str
Initial Amount Paid,17430,80.7,object
Offer Total Amount,17410,80.6,object
Term,9141,42.3,str


**What Deals looks like before cleaning:**
- `Payment Type`, `Months of study`, `Course duration`, `City`, `Level of Deutsch` are filled for a small share
  of deals: managers fill them only at late funnel stages. This is how the process works, not an error.
  A gap means "the deal never reached this stage", so these fields are **neither dropped nor mean-imputed**.
- Amounts are stored as text in different formats: `1000`, `€ 3.500,00`, `$11.000,00`.
- `SLA` mixes `time` (hh:mm:ss) and `timedelta` (more than one day).

In [4]:
raw["Deals"][["Initial Amount Paid", "Offer Total Amount", "SLA", "Level of Deutsch", "City"]].dropna().sample(8, random_state=7)

,Initial Amount Paid,Offer Total Amount,SLA,Level of Deutsch,City
13959,1000,11000,02:09:27,б1,Oberhausen
9628,1000,5000,00:23:45,б1,Binzen
9548,1000,5000,01:00:18,в1,-
14504,1000,11000,10:33:54,б1,Troisdorf
14907,1000,11000,00:00:41,б1,Hagenow
3811,1000,5000,11:17:14,b1,-
11548,1000,5000,17:39:36,В1,Nürnberg
15005,1000,11500,01:21:18,А2 ( хочет просить совмещать),-


In [5]:
missing_report(raw["Calls"])

,missing,%,dtype
Tag,95874,100.0,float64
Dialled Number,95874,100.0,float64
Outgoing Call Status,8999,9.4,str
Scheduled in CRM,8999,9.4,float64
CONTACTID,3933,4.1,str
Call Duration (in seconds),83,0.1,float64


## 2. Cleaning rules and why

| # | Table | Problem | Fix | Reason |
|---|---|---|---|---|
| 1 | Deals | `Lost Reason = Duplicate` | drop | per the FAQ this is a technical duplicate lead; keeping it inflates leads and deflates conversion |
| 2 | Deals | `Closing Date` < `Created Time` | → `NaT` | a deal cannot close before it is created (FAQ: data error, the analyst decides) |
| 3 | Deals | `Initial Amount Paid` > `Offer Total Amount` | 2 rows with offer 1,200 on an 11-month course are a typo in the offer; elsewhere initial is capped at offer | the first payment cannot exceed the course price; conservative revenue estimate |
| 4 | Deals | offer = €0 / 1 / 9 | flag `is_demo` | per the FAQ these are demo accesses for a token fee, not students |
| 5 | Deals | `Payment Done` but `Months of study` empty | not counted as a student | FAQ: "if the month is not filled in, the student is not studying and the money must not be counted" |
| 6 | Deals | amounts as text, SLA of mixed type | → float / seconds | calculations need numbers |
| 7 | Deals | `Level of Deutsch` as free text (Cyrillic б1, в1, "B1 (until February)") | → CEFR `A0…C2` | Cyrillic б/в = B; take the current level (the first one mentioned) |
| 8 | Deals | cities: `-`, street addresses, different spellings | → `city`, `country` | needed for geographic analysis |
| 9 | Deals | `Content`, `Term` | drop | ad-level detail; campaign and source are kept |
| 10 | Deals | categorical gaps | explicit labels `Unknown`, `Not rated`, `Not lost` | the gap itself is informative; dropping rows would distort the funnel |
| 11 | Calls | `Dialled Number`, `Tag` 100% empty | drop | no information |
| 12 | Calls | no duration | drop rows (83) | a call without a duration cannot be analysed |
| 13 | Calls | `Outgoing Call Status` empty for inbound calls | → `Not applicable` | not missing, just not applicable |
| 14 | Spend | 917 full duplicates | drop | all are zero rows and do not affect totals |
| 15 | Spend | `Campaign` empty | → `(no campaign)` | the spend is real and must not be dropped |

**Revenue.** Per the FAQ, payment is confirmed by the status `Payment Done`, and the first payment is `Initial Amount Paid`.
The rest of the course is paid monthly, so the revenue received by the export date is:

$$revenue = initial + (months\_of\_study - 1) \cdot \frac{offer - initial}{course\_duration - 1},\quad revenue \le offer$$

For a one-off payment (`initial = offer`), revenue = `offer`.

In [6]:
clean, logs = etl.run_etl(save=True)
for name, lines in logs.items():
    print(f"\n=== {name}")
    for line in lines:
        print(" •", line)


=== Contacts
 • Duplicate ids removed: 0
 • created_time, modified_time -> datetime64 (format dd.mm.yyyy hh:mm)
 • No missing values; contact_owner_name -> category

=== Calls
 • Duplicate ids removed: 0
 • Fully empty columns dropped: ['dialled_number', 'tag']
 • Calls with missing/negative duration dropped: 83
 • outgoing_call_status: NaN -> 'Not applicable' (inbound/missed); scheduled_in_crm -> bool
 • Calls without contact_id: 3855, kept for manager workload analysis but not joined to deals
 • Added flag is_conversation (>= 30 s): the median call is 8 s, short calls are no-answers/voicemail

=== Spend
 • Full duplicates removed: 917 (all are zero rows with no spend)
 • campaign/adgroup/ad: NaN -> '(no campaign)'/'(not set)': the spend is real, just untagged, so it must not be dropped
 • Negative impressions/spend/clicks values: 0

=== Deals
 • Empty rows/duplicate ids removed: 2
 • Technical duplicate leads removed (Lost Reason = 'Duplicate', see FAQ): 1771
 • Dropped irrelevant c

## 3. Validating the result

In [7]:
pd.DataFrame({
    name: {"rows before": len(raw[name]), "rows after": len(df),
           "columns": df.shape[1], "full duplicates": df.duplicated().sum()}
    for name, df in clean.items()
}).T

,rows before,rows after,columns,full duplicates
Contacts,18548,18548,4,0
Calls,95874,95791,10,0
Spend,20779,19862,8,0
Deals,21595,19822,30,0


In [8]:
deals = clean["Deals"]
deals.dtypes.to_frame("dtype").T

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,source,payment_type,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,deal_cycle_days,sla_sec,amount_anomaly_flag,is_demo,is_paid,is_student,revenue,payments_count,country,german_level,month
dtype,str,category,datetime64[us],category,category,category,str,str,category,category,category,category,datetime64[us],float64,float64,float64,float64,str,str,float64,float64,bool,bool,bool,bool,float64,float64,category,category,datetime64[us]


In [9]:
checks = {
    "deal ids are unique": deals["id"].is_unique,
    "no closing_date < created_time": not (deals["closing_date"] < deals["created_time"].dt.normalize()).any(),
    "initial <= offer everywhere": not (deals["initial_amount_paid"] > deals["offer_total_amount"]).any(),
    "students have months_of_study": deals.loc[deals["is_student"], "months_of_study"].notna().all(),
    "students have a product": deals.loc[deals["is_student"], "product"].ne("Unknown").all(),
    "revenue only for students": (deals.loc[~deals["is_student"], "revenue"] == 0).all(),
    "deal contact_id found in Contacts, %": round(deals["contact_id"].isin(clean["Contacts"]["id"]).mean() * 100, 1),
    "call contact_id found in Contacts, %": round(clean["Calls"]["contact_id"].dropna().isin(clean["Contacts"]["id"]).mean() * 100, 1),
}
pd.Series(checks, name="check").to_frame()

,check
deal ids are unique,True
no closing_date < created_time,True
initial <= offer everywhere,True
students have months_of_study,True
students have a product,True
revenue only for students,True
"deal contact_id found in Contacts, %",99.8
"call contact_id found in Contacts, %",100.0


In [10]:
deals["german_level"].value_counts().to_frame("deals")

,deals
german_level,
Unknown,18623
B1,815
B2,169
A2,149
C1,27
A1,26
A0,10
C2,3


## 4. Summary

- Deals after cleaning: **19,822** (1,771 technical duplicates removed), calls: **95,791**, spend rows: **19,862**.
- Students (paid + study months filled in, not demo): **837** of 856 `Payment Done` deals.
- The joins work: 99.7% of deals and 100% of calls with a contact find their contact.
- Clean tables are saved to `data/processed/` in two formats: **CSV** (for Tableau/Sheets) and
  **pickle** (for Python: keeps the `datetime`, `category` and `bool` types and loads instantly).

➡️ Next step: [02_eda_analysis.ipynb](02_eda_analysis.ipynb)